# 梯度提升树（GBDT）原理解析与完整实战

> 从 XGBoost / LightGBM / CatBoost 的学起  
> 包含数学公式 + scikit-learn 实现 + 现代高效实现推荐

---

## 模块一：梯度提升树核心原理
### 1.1 Boosting vs Bagging 对比

| 项目         | Bagging（如随机森林）     | Boosting（如 GBDT）             |
|--------------|----------------------------|----------------------------------|
| 训练方式     | 并行、独立                 | 串行、依赖前一棵树               |
| 样本权重     | 有放回等概率抽样           | 错分样本权重增大                 |
| 目标         | 降低方差（Variance）       | 降低偏差（Bias）                 |
| 典型代表     | Random Forest              | GBDT、XGBoost、LightGBM、CatBoost |

### 1.2 GBDT 核心数学公式

GBDT 是 **函数空间的梯度下降**：

$$
\boxed{\hat{y} = f_M(x) = \sum_{m=1}^M \alpha_m h_m(x)}
$$

其中：
- $ h_m(x) $：第 m 棵弱学习器（通常是 CART 回归树）
- $ \alpha_m $：第 m 棵树的权重（通常为学习率 η）

每一步拟合的是**当前模型的负梯度（残差）**：

分类（对数损失）：
$$
r_{im} = y_i - \frac{1}{1+e^{f_{m-1}(x_i)}}
$$

回归（平方损失）：
$$
\boxed{r_{im} = y_i - f_{m-1}(x_i)} \quad \text{（就是残差！）}
$$

学习率（shrinkage）：
$$
f_m(x) = f_{m-1}(x) + \eta \cdot h_m(x), \quad \eta \in (0,1]
$$

### 1.3 三种主流现代实现对比

| 算法       | 速度     | 精度     | 对类别特征 | 推荐指数 |
|------------|----------|----------|------------|----------|
| sklearn GBDT | 慢     | 一般     | 不支持     | ★★       |
| XGBoost    | 快      | 极高     | 需要one-hot| ★★★★★    |
| LightGBM   | 极快    | 极高     | 原生支持   | ★★★★★★   |
| CatBoost   | 快      | 极高     | 原生支持+抗过拟合 | ★★★★★★ |

---

## 模块二：sklearn 原生 GBDT

In [1]:
# 模块二：sklearn 原生 GradientBoosting
from sklearn.ensemble import GradientBoostingClassifier, GradientBoostingRegressor
from sklearn.datasets import load_breast_cancer, fetch_california_housing
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, mean_squared_error, r2_score
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

plt.rcParams['font.sans-serif'] = ['SimHei', 'Arial Unicode MS']
plt.rcParams['axes.unicode_minus'] = False

In [2]:
# 分类任务：乳腺癌数据集
cancer = load_breast_cancer()
X, y = cancer.data, cancer.target
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

gbdt_clf = GradientBoostingClassifier(
    n_estimators=500,        # 树的数量
    learning_rate=0.05,      # 学习率（关键参数！）
    max_depth=3,             # 树不能太深
    subsample=0.8,           # 随机子采样（引入随机性，防过拟合）
    random_state=42
)

gbdt_clf.fit(X_train, y_train)
print(f"GBDT 分类准确率: {accuracy_score(y_test, gbdt_clf.predict(X_test)):.4f}")

GBDT 分类准确率: 0.9561


In [3]:
# 回归任务：加州房价
housing = fetch_california_housing()
X_reg, y_reg = housing.data, housing.target
X_train_r, X_test_r, y_train_r, y_test_r = train_test_split(X_reg, y_reg, test_size=0.2, random_state=42)

gbdt_reg = GradientBoostingRegressor(
    n_estimators=800,
    learning_rate=0.05,
    max_depth=4,
    subsample=0.8,
    random_state=42
)

gbdt_reg.fit(X_train_r, y_train_r)
pred = gbdt_reg.predict(X_test_r)
print(f"GBDT 回归 RMSE: {np.sqrt(mean_squared_error(y_test_r, pred)):.4f}")
print(f"GBDT 回归 R²: {r2_score(y_test_r, pred):.4f}")

GBDT 回归 RMSE: 0.4668
GBDT 回归 R²: 0.8337


## 模块三：现代高效实现 推荐日常使用

In [4]:

import lightgbm as lgb
import xgboost as xgb
from catboost import CatBoostClassifier, CatBoostRegressor

# LightGBM（最快 + 原生支持类别特征）
lgb_clf = lgb.LGBMClassifier(
    n_estimators=1000,
    learning_rate=0.05,
    num_leaves=31,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    n_jobs=-1
)
lgb_clf.fit(X_train, y_train)
print(f"LightGBM 准确率: {accuracy_score(y_test, lgb_clf.predict(X_test)):.4f}")

# XGBoost
xgb_clf = xgb.XGBClassifier(
    n_estimators=1000,
    learning_rate=0.05,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    n_jobs=-1,
    verbosity=0
)
xgb_clf.fit(X_train, y_train)
print(f"XGBoost 准确率: {accuracy_score(y_test, xgb_clf.predict(X_test)):.4f}")

# CatBoost（对类别特征最友好，几乎免调参）
cat_clf = CatBoostClassifier(
    iterations=1000,
    learning_rate=0.05,
    depth=6,
    random_seed=42,
    verbose=100
)
cat_clf.fit(X_train, y_train)
print(f"CatBoost 准确率: {accuracy_score(y_test, cat_clf.predict(X_test)):.4f}")

[LightGBM] [Info] Number of positive: 285, number of negative: 170
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000880 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 4542
[LightGBM] [Info] Number of data points in the train set: 455, number of used features: 30
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.626374 -> initscore=0.516691
[LightGBM] [Info] Start training from score 0.516691
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best 

D:\anaconda\envs\financial-ml\lib\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


XGBoost 准确率: 0.9649
0:	learn: 0.6141452	total: 135ms	remaining: 2m 14s
100:	learn: 0.0161972	total: 386ms	remaining: 3.44s
200:	learn: 0.0062872	total: 616ms	remaining: 2.45s
300:	learn: 0.0049004	total: 832ms	remaining: 1.93s
400:	learn: 0.0041577	total: 1.06s	remaining: 1.59s
500:	learn: 0.0038361	total: 1.3s	remaining: 1.3s
600:	learn: 0.0035870	total: 1.53s	remaining: 1.02s
700:	learn: 0.0034714	total: 1.74s	remaining: 742ms
800:	learn: 0.0033490	total: 1.98s	remaining: 492ms
900:	learn: 0.0032229	total: 2.34s	remaining: 257ms
999:	learn: 0.0031763	total: 2.62s	remaining: 0us
CatBoost 准确率: 0.9561


## 模块四：GBDT 超参数调优速查表（适用于 LightGBM/XGBoost/CatBoost）



| 参数                  | 推荐起始值         | 调优建议                                                                 |
|-----------------------|--------------------|--------------------------------------------------------------------------|
| n_estimators          | 500~5000           | 配合 early_stopping 使用，几乎不用手动设太大                              |
| learning_rate         | 0.01~0.1           | 越小越好（推荐 0.05 开始），越小则 n_estimators 相应增大                  |
| max_depth / depth     | 3~10               | LightGBM 建议改用 num_leaves 控制树复杂度                                |
| num_leaves            | 31~128 (LightGBM)  | 推荐值 ≈ 2^(max_depth)，常用 31、64、128                                 |
| subsample             | 0.6~1.0            | <1.0 引入随机性，加速训练并防过拟合，推荐 0.8                            |
| colsample_bytree      | 0.6~1.0            | 每棵树随机采样特征比例，推荐 0.6~0.9                                     |
| early_stopping_rounds | 50~100             | 防止过拟合，必开！验证集指标连续 n 轮不提升就停止训练                     |
| min_child_weight      | 1~10 (XGBoost)     | 控制叶子节点最小权重，越大越保守                                         |
| reg_alpha / reg_lambda| 0~1.0              | L1/L2 正则化，防止过拟合，轻度使用即可                                   |

### 常用调参顺序（工业界/比赛通用流程）

1. 先固定 `learning_rate=0.1`，用早停找到大致需要的 `n_estimators`
2. 降低 `learning_rate` 到 0.05 或 0.01，同时将 `n_estimators` 乘以 3~10 倍
3. 调节树复杂度：`max_depth` / `num_leaves` / `min_child_weight`
4. 调节采样：`subsample` + `colsample_bytree`（

## 模块五：最终建议

| 场景                         | 推荐算法         | 理由                                      |
|------------------------------|------------------|-------------------------------------------|
| 结构化/表格数据（<10万行）   | LightGBM         | 速度快、精度高、原生支持类别特征           |
| 有大量类别特征               | CatBoost         | 自动处理类别特征，几乎免调参               |
| 需要最高精度 + 可解释性      | XGBoost + SHAP   | 支持 SHAP 值解释                          |
